[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance1_cours.ipynb)

# Séance 3.1 — Décrire une distribution

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire pourquoi une moyenne seule est presque toujours trompeuse
- choisir entre moyenne et médiane selon la forme de la distribution
- lire un `describe()` ligne par ligne
- mesurer la dispersion avec l'écart-type et l'écart interquartile
- repérer une concentration : quelle part du total tient dans le haut du classement

## Une phrase de rapport, et le problème

> *« Le panier moyen de nos clients est de 590 €. »*

Cette phrase se trouve dans à peu près tous les rapports d'activité. Elle a
l'air d'une information. Posez-vous la question suivante :

**Si vous deviez fixer le seuil de livraison gratuite, le mettriez-vous à
590 € ?**

À la fin de cette séance vous saurez pourquoi la réponse est non, et ce qu'il
fallait regarder à la place.

### Les données

Même détaillant que le bloc 2, mais à une **maille** différente : une ligne
n'est plus un produit vendu, c'est **une commande entière**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

print(cmd.shape)
cmd.head(3)

`ca` est le montant total de la commande, `nart` son nombre de références
différentes, `qte` son nombre d'articles.

## 1. La moyenne n'est pas le milieu

Deux façons de dire « la valeur typique » :

- la **moyenne** : on additionne tout, on divise par le nombre
- la **médiane** : on classe, on prend celle du milieu — la moitié en dessous,
  la moitié au-dessus

In [ ]:
print("moyenne :", round(cmd["ca"].mean(), 2))
print("mediane :", round(cmd["ca"].median(), 2))

**590 € contre 356 €.** Un écart de 234 €, soit deux tiers de la médiane. Ces
deux nombres décrivent le même fichier.

Question avant d'exécuter la cellule suivante : à votre avis, quelle
proportion des commandes dépasse la « moyenne » ?

In [ ]:
part = 100 * (cmd["ca"] > cmd["ca"].mean()).mean()

print(round(part, 1), "% des commandes depassent la moyenne")

**29 %.** Sept commandes sur dix sont en dessous de la « moyenne ». Un seuil
de livraison gratuite à 590 € serait hors de portée pour 71 % des commandes.

Regardons pourquoi.

In [ ]:
cmd["ca"].plot(kind="hist", bins=40, figsize=(7, 4))
plt.title("Repartition des paniers")
plt.xlabel("montant de la commande (euros)")
plt.show()

Voilà la forme : un **tassement à gauche** et une **traîne qui s'étire loin à
droite**. Quelques commandes énormes (jusqu'à 16 775 €) tirent la moyenne vers
le haut sans déplacer la médiane d'un centime.

> 💡 **La règle.** Distribution symétrique : moyenne et médiane coïncident,
> prenez l'une ou l'autre. Distribution asymétrique — et **presque tout ce qui
> est en euros l'est** — la médiane décrit la situation typique, la moyenne
> décrit le total divisé par l'effectif. Affichez les deux.

## 2. Tout voir d'un coup : `describe()`

In [ ]:
cmd["ca"].describe().round(2)

Huit nombres, et la distribution est décrite :

| Ligne | Ce que ça dit ici |
|---|---|
| `count` | 1 955 commandes |
| `mean` | 589,73 € — la moyenne |
| `std` | 918,41 € — l'écart-type, voir plus bas |
| `min` | 1,45 € — la plus petite commande |
| `25%` | 189,65 € — un quart des commandes sont en dessous |
| `50%` | 355,89 € — la médiane |
| `75%` | 659,52 € — trois quarts sont en dessous |
| `max` | 16 774,72 € — la plus grosse |

### Les quantiles répondent aux questions de seuil

« À partir de quel montant une commande fait-elle partie des 10 % les plus
grosses ? » se lit directement :

In [ ]:
print("seuil des 10 % du haut :", round(cmd["ca"].quantile(0.9), 2))
print("seuil des 10 % du bas  :", round(cmd["ca"].quantile(0.1), 2))

### Mesurer la dispersion

Deux commandes à 350 € et 360 €, ou deux commandes à 10 € et 700 € : même
moyenne, situation très différente. La **dispersion** mesure cet étalement.

In [ ]:
q1 = cmd["ca"].quantile(0.25)
q3 = cmd["ca"].quantile(0.75)

print("ecart-type          :", round(cmd["ca"].std(), 2))
print("ecart interquartile :", round(q3 - q1, 2))

L'**écart-type** (`std`) est l'écart typique à la moyenne. Il se lit dans la
même unité que les données — ici des euros.

**Regardez-le bien : 918 €, soit plus que la moyenne elle-même.** Quand
l'écart-type dépasse la moyenne, il n'y a pas de valeur typique. Dire « le
panier moyen est de 590 € » est alors une phrase vide.

L'**écart interquartile** (q3 − q1, ici 470 €) est sa version robuste : il
décrit l'étalement de la moitié centrale et ignore les extrêmes. Il ne bouge
pas si la plus grosse commande double.

## 3. Où est concentré le chiffre d'affaires ?

Une traîne à droite pose toujours la même question business : **quelle part du
total tient dans le haut du classement ?**

In [ ]:
top = cmd["ca"].sort_values(ascending=False)
n10 = int(0.10 * len(cmd))   # les 10 % de commandes les plus grosses

part = 100 * top.head(n10).sum() / top.sum()
print(n10, "commandes font", round(part, 1), "% du chiffre d'affaires")

**195 commandes sur 1 955 font 41,3 % du chiffre d'affaires.**

Vous avez déjà croisé ce phénomène en séance 2.3 : deux clients irlandais
pesaient 22,7 % du CA. Ce n'était pas une anomalie isolée, c'est la façon
dont ce marché est fait. Une moyenne, par construction, écrase exactement
cette information.

## 4. Comparer des groupes sans se faire piéger

Même question, pays par pays. Notez l'ordre des colonnes : **`count` d'abord**.

In [ ]:
parpays = cmd.groupby("pays")["ca"].agg(["count", "mean", "median"])

parpays.query("count >= 20").sort_values("mean", ascending=False).round(2)

Trois lectures :

- **La Suède : 1 409 € de moyenne, 486 € de médiane.** L'écart le plus violent
  du tableau, sur 25 commandes. Une ou deux commandes exceptionnelles suffisent
  à produire ce chiffre.
- **L'Irlande : 1 020 € de moyenne** — et vous savez depuis la séance 2.3
  qu'elle n'a que **deux clients**.
- **Le Royaume-Uni : moyenne 394 €, médiane 301 €.** L'écart le plus faible :
  c'est le marché le plus régulier, celui sur lequel une moyenne veut dire
  quelque chose.

> ⚠️ **Le filtre `count >= 20` n'est pas de la coquetterie.** Sans lui, le
> Japon arrive en tête du classement avec 2 178 € de moyenne… sur 12 commandes.
> Regardez toujours l'effectif avant de commenter un groupe.

In [ ]:
parpays.query("count >= 20").sort_values("median")["median"].plot(
    kind="barh", figsize=(7, 4))
plt.title("Panier median par pays")
plt.xlabel("euros")
plt.show()

## 5. Deux erreurs, dont une qui ne prévient pas

### L'erreur bruyante

In [ ]:
cmd["jour"].mean()

Dernière ligne :

```
TypeError: Could not convert string 'mercredi...' to numeric
```

Traduction : on a demandé une moyenne sur du texte. Rien à réparer, la
question n'avait pas de sens. **Seule la dernière ligne d'une erreur compte.**

### L'erreur silencieuse

Celle-ci tourne, ne prévient de rien, et donne un chiffre faux.

In [ ]:
fausse = cmd.groupby("pays")["ca"].mean().mean()   # moyenne DES MOYENNES
vraie = cmd["ca"].mean()                            # moyenne des commandes

print("moyenne des moyennes :", round(fausse, 2))
print("moyenne reelle       :", round(vraie, 2))

**750,88 € contre 589,73 €.** Une erreur de +27 %, dans un chiffre qui
s'affiche sans le moindre avertissement.

Pourquoi ? La moyenne des moyennes traite le Canada — **une** commande —
exactement comme le Royaume-Uni et ses 798 commandes. Chaque pays pèse un
vingt-troisième du résultat, quel que soit son poids réel.

> ⚠️ **La règle : on ne fait jamais la moyenne de moyennes.** Pour un
> indicateur global, on repart toujours des données individuelles.

Laquelle des deux erreurs est la plus dangereuse ? La première vous a arrêté.
La seconde serait partie dans une présentation.

## 6. Ce que les données ne disent pas

In [ ]:
cmd["jour"].value_counts()

Comptez les lignes : **six jours**. Le samedi n'apparaît pas.

Ce n'est pas un zéro affiché, c'est une **absence** — et une absence ne se
voit dans aucun `describe()`, aucune moyenne, aucun graphique par jour.
L'enseigne ne traite aucune commande le samedi.

Conséquence pratique : un chiffre d'affaires « moyen par jour » calculé en
divisant par 7 est faux de 17 %. Avant de diviser par un nombre de jours,
vérifiez le nombre de jours qui existent vraiment.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| tout d'un coup | `df["ca"].describe()` |
| le centre, version fragile | `df["ca"].mean()` |
| le centre, version robuste | `df["ca"].median()` |
| un seuil, ici les 10 % du haut | `df["ca"].quantile(0.9)` |
| la dispersion, version fragile | `df["ca"].std()` |
| la dispersion, version robuste | `q3 - q1` |
| la forme | `df["ca"].plot(kind="hist", bins=40)` |
| comparer des groupes | `df.groupby("pays")["ca"].agg(["count", "mean", "median"])` |
| combien de modalités | `df["jour"].nunique()` |

## Les trois réflexes de la séance

1. **Toujours afficher la médiane à côté de la moyenne.** L'écart entre les
   deux mesure l'asymétrie. Ici : 590 € contre 356 €, et seulement 29 % des
   commandes dépassent la « moyenne ».

2. **Ne jamais faire la moyenne de moyennes.** Elle donne le même poids à un
   pays qui pèse 798 commandes et à un pays qui en pèse 12. Toujours repartir
   des données individuelles.

3. **Compter les modalités avant de commenter un groupe.** Une moyenne sur
   12 commandes n'est pas un résultat, c'est une anecdote.